# TP - Classification de cartes magnétiques
## Détection de fourreaux souterrains par apprentissage profond
**Formation :** 2ème année Data IA  
**Date de rendu :** 11 juin 2025 à 17h00

## Installation des dépendances

In [ ]:
# Installe les bibliothèques nécessaires
!pip install tensorflow numpy pandas matplotlib seaborn scikit-learn opencv-python pillow -q

In [ ]:
# Monte Google Drive et définit le chemin des données
from google.colab import drive
drive.mount("/content/drive")

# Adapte ce chemin selon l'emplacement du dossier dans ton Drive
DATA_DIR = "/content/drive/MyDrive/data"  # ← modifie si besoin

## Connexion Google Drive (Colab)

In [ ]:
# Split stratifié 70/15/15
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

mean, std = compute_channel_stats(X_train)
X_train = normalize(X_train, mean, std)
X_val   = normalize(X_val, mean, std)
X_test  = normalize(X_test, mean, std)

n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()
class_weight = {0: 1.0, 1: n_neg / n_pos}

print(f"Train : {X_train.shape}  Val : {X_val.shape}  Test : {X_test.shape}")
print(f"Class weights : {class_weight}")

In [ ]:
IMG_SIZE = 128


def fill_nan_median(data):
    """Remplace NaN par médiane canal."""
    result = data.copy()
    for c in range(data.shape[2]):
        channel = result[:, :, c]
        channel[np.isnan(channel)] = np.nanmedian(channel)
    return result


def resize_map(data, size=IMG_SIZE):
    """Redimensionne la carte magnétique."""
    resized = np.zeros((size, size, 4), dtype=np.float32)
    for c in range(4):
        resized[:, :, c] = cv2.resize(
            data[:, :, c], (size, size), interpolation=cv2.INTER_LINEAR
        )
    return resized


def compute_channel_stats(X):
    """Calcule mean/std sur train set."""
    mean = X.mean(axis=(0, 1, 2), keepdims=True)
    std = X.std(axis=(0, 1, 2), keepdims=True) + 1e-8
    return mean, std


def normalize(X, mean, std):
    """Applique z-score normalisation."""
    return (X - mean) / std


def preprocess_sample(fname):
    """Charge, nettoie et redimensionne un échantillon."""
    data = load_npz(fname)
    data = fill_nan_median(data)
    data = resize_map(data)
    return data


print("Chargement de tous les échantillons...")
X = np.array([preprocess_sample(f) for f in df["field_file"]], dtype=np.float32)
y = df["label"].values.astype(np.float32)
print(f"X : {X.shape}  |  y : {y.shape}")

## 2) Prétraitement des données

### Gestion des NaN
Remplacement par la **médiane du canal** : robuste aux outliers, préserve l'ordre de grandeur physique du signal magnétique.

### Redimensionnement
Taille cible **128×128** : assez grand pour conserver les structures dipôlaires, assez petit pour rester léger (~3000 échantillons).

### Normalisation
**Z-score par canal**, statistiques calculées sur le train set uniquement (pas de data leakage).

**Observations :**
- Les cartes **avec fourreau** présentent une anomalie dipôlaire marquée (alternance rouge/bleu intense sur Bz et Btotal).
- Les cartes **sans fourreau** montrent un signal homogène de faible amplitude, dominé par du bruit diffus.
- La composante **Btotal** est la plus discriminante visuellement.

In [ ]:
CHANNEL_NAMES = ["Bx (nT)", "By (nT)", "Bz (nT)", "Btotal (nT)"]


def plot_sample(fname, label, ax_row):
    """Affiche les 4 canaux d'un échantillon."""
    data = load_npz(fname)
    title_prefix = "AVEC fourreau" if label == 1 else "SANS fourreau"
    for c, ax in enumerate(ax_row):
        channel = data[:, :, c]
        vmax = np.nanpercentile(np.abs(channel), 99)
        im = ax.imshow(channel, cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto")
        ax.set_title(f"{title_prefix}\n{CHANNEL_NAMES[c]}", fontsize=8)
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        ax.axis("off")


samples_pos = df[df["label"] == 1]["field_file"].iloc[:2].tolist()
samples_neg = df[df["label"] == 0]["field_file"].iloc[:2].tolist()
samples = [(f, 1) for f in samples_pos] + [(f, 0) for f in samples_neg]

fig, axes = plt.subplots(4, 4, figsize=(16, 14))
for row, (fname, lbl) in enumerate(samples):
    plot_sample(fname, lbl, axes[row])

plt.suptitle("Cartes magnétiques — 4 canaux", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
def load_npz(filename):
    """Charge un fichier npz."""
    path = os.path.join(DATA_DIR, filename)
    sample = np.load(path, allow_pickle=True)
    return sample["data"].astype(np.float32)


# Inspecte les 8 premiers fichiers
for fname in df["field_file"].iloc[:8]:
    data = load_npz(fname)
    nan_pct = np.isnan(data).mean() * 100
    print(f"{fname:<60} shape={data.shape}  nan={nan_pct:.1f}%  "
          f"min={np.nanmin(data):.2f}  max={np.nanmax(data):.2f}")

In [ ]:
# Visualise la distribution des classes
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

df["label"].value_counts().plot(kind="bar", ax=axes[0], color=["#2196F3", "#F44336"])
axes[0].set_title("Distribution des labels")
axes[0].set_xticklabels(["Absent (0)", "Présent (1)"], rotation=0)

df["coverage_type"].value_counts().plot(kind="bar", ax=axes[1])
axes[1].set_title("Coverage type")

df["noise_type"].value_counts().plot(kind="bar", ax=axes[2])
axes[2].set_title("Noise type")

plt.tight_layout()
plt.show()

In [ ]:
# Distribution des classes
print("Distribution des labels :")
print(df["label"].value_counts())
print(f"\nTaux de déséquilibre : {df['label'].value_counts()[1] / len(df):.2%} positifs")

In [ ]:
CSV_PATH = os.path.join(DATA_DIR, "pipe_presence_width_detection_label.csv")

# Charge le fichier labels
df = pd.read_csv(CSV_PATH, sep=";")
print("Shape:", df.shape)
df.head()

## 1) Chargement et exploration du dataset

---
# Partie I — Exploration et prétraitement des données

## Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_curve, auc, accuracy_score,
    precision_score, recall_score, f1_score
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.applications import EfficientNetB0

print(f"TensorFlow : {tf.__version__}")
print(f"GPU disponible : {tf.config.list_physical_devices('GPU')}")